# JEPA-for-Trading V3: abstention world model planner

V3 starts from the valid V2 world model idea, but fixes the biggest trading failure observed in the V2 run: the planner traded too aggressively and did not learn a strong enough no-trade/risk-off behavior.

This notebook trains a V3 world model where every sample contains multiple candidate actions: hold, cash, de-risk, equal weight, volatility target, and sampled actions. The model learns outcomes and utility ranking across those candidates, then the planner executes a trade only when it beats hold/cash by a margin.


In [ ]:
# Kaggle/bootstrap cell. Run this first.
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/aurvl/jepa-for-trading.git"
BRANCH = "version3"
REPO_DIR = Path("/kaggle/working/jepa-for-trading") if Path("/kaggle/working").exists() else Path.cwd()

if not (REPO_DIR / "src" / "jepa_trading").exists():
    if REPO_DIR.exists():
        print(f"Repo dir exists but package not found: {REPO_DIR}")
    else:
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
!{sys.executable} -m pip install -q -e . --no-deps

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from jepa_trading.config import ensure_dirs, load_config
from jepa_trading.data.pipeline import prepare_market_data, create_v3_dataloaders
from jepa_trading.data.v2_dataset import PortfolioActionConfig
from jepa_trading.data.v3_dataset import summarize_v3_batch
from jepa_trading.evaluation.backtest import (
    buy_and_hold_weight,
    equal_weight,
    momentum_weight,
    random_long_only_weight,
    run_weight_strategy,
    volatility_target_weight,
)
from jepa_trading.evaluation.metrics import metrics_table, validate_backtest_histories
from jepa_trading.evaluation.plots import plot_drawdown, plot_equity_curves, plot_turnover
from jepa_trading.evaluation.statistical_tests import bootstrap_mean_return_p_value, randomization_p_value
from jepa_trading.evaluation.v3_backtest import run_v3_planner_backtest
from jepa_trading.models.world_model_v2 import V2WorldModel
from jepa_trading.planning.v3_planner import V3AbstentionPlanner
from jepa_trading.rl.env import TradingEnv
from jepa_trading.rl.observer import RawMarketObserver
from jepa_trading.training.checkpoints import load_checkpoint
from jepa_trading.training.train_v3 import train_v3_world_model
from jepa_trading.utils.device import get_device
from jepa_trading.utils.seed import seed_everything


In [ ]:
config = load_config("configs/default.yaml")

# Kaggle: auto-discover macro file. CSV is accepted too.
macro_candidates = []
if Path("/kaggle/input").exists():
    macro_candidates += list(Path("/kaggle/input").rglob("macro_data.parquet"))
    macro_candidates += list(Path("/kaggle/input").rglob("estimated_volatility_with_macro.csv"))
if macro_candidates:
    config["data"]["macro_path"] = str(macro_candidates[0])
    print("Using macro file:", config["data"]["macro_path"])
else:
    print("Using configured macro file:", config["data"]["macro_path"])

FAST_DEV_RUN = False
if FAST_DEV_RUN:
    config["data"]["tickers"] = config["data"]["tickers"][:8]
    config["v3"]["world_max_steps"] = 50
    config["v3"]["world_warmup_steps"] = 10
    config["v3"]["planner_sampled_actions"] = 16

ensure_dirs(config)
seed_everything(config["seed"])
device = get_device(config["device"])
device


In [ ]:
prepared_df, arrays, feature_columns = prepare_market_data(config, force_download=False)
loaders = create_v3_dataloaders(config, arrays)

print("rows:", len(prepared_df))
print("assets:", len(arrays.tickers), list(arrays.tickers))
print("features:", len(feature_columns), feature_columns)
print("dates:", arrays.dates.min(), "->", arrays.dates.max())
print("dataset sizes:", {k: len(v.dataset) for k, v in loaders.items()})

batch = next(iter(loaders["train"]))
print("batch summary:", summarize_v3_batch(batch))


In [ ]:
portfolio_cfg = config["portfolio"]
v3_cfg = config["v3"]

model = V2WorldModel(
    n_features=len(feature_columns),
    max_assets=len(arrays.tickers),
    portfolio_state_dim=batch["portfolio_state"].shape[-1],
    action_dim=batch["actions"].shape[-1],
    d_model=config["model"]["d_model"],
    latent_dim=config["model"]["latent_dim"],
    n_heads=config["model"]["n_heads"],
    n_layers=config["model"]["n_layers"],
    dropout=config["model"]["dropout"],
    ema_decay=config["training"]["ema_decay"],
    hidden_dim=config["v2"].get("hidden_dim", 256),
    policy_mode=portfolio_cfg.get("mode", "long_only"),
    max_abs_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    asset_embedding_dim=config["model"].get("asset_embedding_dim"),
)

v3_ckpt = Path(config["training"]["checkpoint_dir"]) / "v3_world_model.pt"
if v3_ckpt.exists():
    print("Loading existing V3 checkpoint:", v3_ckpt)
    metadata = load_checkpoint(v3_ckpt, model, map_location=device)
    print("checkpoint metadata:", metadata)
else:
    v3_history = train_v3_world_model(
        model=model,
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        device=device,
        max_steps=v3_cfg["world_max_steps"],
        warmup_steps=v3_cfg["world_warmup_steps"],
        lr=config["training"]["lr"],
        weight_decay=config["training"]["weight_decay"],
        checkpoint_path=v3_ckpt,
        weights=v3_cfg["loss_weights"],
        rank_margin=v3_cfg["rank_margin"],
        eval_every=config["training"]["eval_every"],
        log_every=config["training"]["log_every"],
    )
    display(v3_history.tail())
    load_checkpoint(v3_ckpt, model, map_location=device)

model.to(device).eval()
print("model ready")


In [ ]:
action_cfg = PortfolioActionConfig(
    mode=portfolio_cfg.get("mode", "long_only"),
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
    n_action_samples=v3_cfg["planner_sampled_actions"],
    derisk_fraction=portfolio_cfg.get("derisk_fraction", 0.50),
)

planner = V3AbstentionPlanner(
    model=model,
    action_config=action_cfg,
    horizons=config["data"]["horizons"],
    n_sampled_actions=v3_cfg["planner_sampled_actions"],
    no_trade_margin=v3_cfg["no_trade_margin"],
    cash_margin=v3_cfg["cash_margin"],
    turnover_energy_penalty=v3_cfg["turnover_energy_penalty"],
    risk_off_drawdown_threshold=v3_cfg["risk_off_drawdown_threshold"],
    chunk_size=256,
    device=device,
    seed=config["seed"] + 300,
)

start_date = pd.Timestamp(config["data"]["val_end"]) + pd.offsets.BDay(1)
end_date = pd.Timestamp(arrays.dates[-2])

common_env_kwargs = dict(
    arrays=arrays,
    observer=RawMarketObserver(arrays, config["data"]["lookback"]),
    start_date=start_date,
    end_date=end_date,
    lookback=config["data"]["lookback"],
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_weight_per_asset=portfolio_cfg.get("max_weight_per_asset", portfolio_cfg.get("max_long_weight", 0.15)),
    max_turnover=portfolio_cfg["max_turnover"],
    mode=portfolio_cfg.get("mode", "long_only"),
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
)

agent_history = run_v3_planner_backtest(
    arrays=arrays,
    planner=planner,
    start_date=start_date,
    end_date=end_date,
    lookback=config["data"]["lookback"],
    cash_initial=portfolio_cfg["cash_initial"],
    transaction_cost_bps=portfolio_cfg["transaction_cost_bps"],
    max_weight_per_asset=portfolio_cfg.get("max_weight_per_asset", portfolio_cfg.get("max_long_weight", 0.15)),
    max_turnover=portfolio_cfg["max_turnover"],
    mode=portfolio_cfg.get("mode", "long_only"),
    max_long_weight=portfolio_cfg.get("max_long_weight", portfolio_cfg.get("max_weight_per_asset", 0.15)),
    max_short_weight=portfolio_cfg.get("max_short_weight", 0.05),
    max_gross_exposure=portfolio_cfg.get("max_gross_exposure", 1.0),
    max_net_exposure=portfolio_cfg.get("max_net_exposure", 1.0),
    borrow_cost_bps=portfolio_cfg.get("borrow_cost_bps", 2.0),
)

print(agent_history.tail())
print(agent_history["selected_action_name"].value_counts(normalize=True))


In [ ]:
def make_env():
    return TradingEnv(**common_env_kwargs)

buy_hold = run_weight_strategy(make_env(), buy_and_hold_weight)
equal_hist = run_weight_strategy(make_env(), equal_weight)
momentum_hist = run_weight_strategy(make_env(), lambda env, mask: momentum_weight(env, mask, lookback=20))
vol_target_hist = run_weight_strategy(make_env(), volatility_target_weight)

N_RANDOM = 100
random_histories = []
for i in range(N_RANDOM):
    rng = np.random.default_rng(config["seed"] + 1000 + i)
    random_histories.append(run_weight_strategy(make_env(), lambda env, mask, rng=rng: random_long_only_weight(env, mask, rng)))

histories = {
    "V3 JEPA Abstention Planner": agent_history,
    "Buy & Hold": buy_hold,
    "Equal Weight": equal_hist,
    "Momentum": momentum_hist,
    "Vol Target": vol_target_hist,
}
validity = validate_backtest_histories(histories)
display(validity)

metrics = metrics_table(histories)
display(metrics)

if not validity["valid_backtest"].all():
    print("INVALID BACKTEST: do not interpret performance positively.")
else:
    random_p = randomization_p_value(agent_history, random_histories)
    boot_vs_bh = bootstrap_mean_return_p_value(agent_history, buy_hold)
    print("p_value_random_beats_agent:", random_p)
    print("bootstrap agent minus buy_hold:", boot_vs_bh)


In [ ]:
plot_equity_curves(
    agent_history,
    buy_hold,
    random_histories,
    extra={"Equal Weight": equal_hist, "Momentum": momentum_hist, "Vol Target": vol_target_hist},
)
plot_drawdown(agent_history, label="V3 JEPA Abstention Planner")
plot_turnover(agent_history)

fig, ax = plt.subplots(figsize=(14, 4))
agent_history["selected_action_name"].value_counts().plot(kind="bar", ax=ax, color="#39ff14")
ax.set_title("V3 selected action types")
ax.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(agent_history["date"], agent_history["cash_weight"], color="#39ff14")
ax.set_title("V3 cash weight over time")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Final cell: push the run artifacts/results back to GitHub branch version3.
# Requires a Kaggle Secret named GITHUB_TOKEN with repo write access.
import os
import subprocess
from pathlib import Path

BRANCH = "version3"
REMOTE_REPO = "github.com/aurvl/jepa-for-trading.git"
GIT_USER_NAME = "aurvl"
GIT_USER_EMAIL = "aurelvhei@outlook.fr"
COMMIT_MESSAGE = "Add V3 Kaggle run artifacts"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = os.environ.get("GITHUB_TOKEN", "")

if not token:
    raise RuntimeError("Missing GITHUB_TOKEN. Add it in Kaggle Secrets before running this cell.")

def run(cmd, check=True):
    printable = " ".join(cmd)
    if token:
        printable = printable.replace(token, "TOKEN_REDACTED")
    print("$", printable)
    proc = subprocess.run(cmd, text=True, capture_output=True)
    out = (proc.stdout or "").replace(token, "TOKEN_REDACTED")
    err = (proc.stderr or "").replace(token, "TOKEN_REDACTED")
    if out.strip():
        print(out)
    if err.strip():
        print(err)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=out, stderr=err)
    return proc

run(["git", "config", "user.name", GIT_USER_NAME])
run(["git", "config", "user.email", GIT_USER_EMAIL])
run(["git", "checkout", BRANCH], check=False)

# Track large model artifacts with LFS when available.
run(["git", "lfs", "install"], check=False)
run(["git", "lfs", "track", "models/*.pt"], check=False)
run(["git", "lfs", "track", "outputs/**"], check=False)

run(["git", "add", "."])
# These folders can be ignored by default; force-add only if they exist and you really want the run outputs.
for path in ["models", "outputs", "logs"]:
    if Path(path).exists():
        run(["git", "add", "-f", path], check=False)

status = run(["git", "status", "--short"], check=False)
if status.stdout.strip():
    run(["git", "commit", "-m", COMMIT_MESSAGE], check=False)
else:
    print("Nothing to commit; pushing current HEAD.")

auth_prefix = "https://" + "x-access-token" + ":"
auth_remote = f"{auth_prefix}{token}@{REMOTE_REPO}"
run(["git", "remote", "set-url", "origin", auth_remote])
run(["git", "push", "origin", f"HEAD:{BRANCH}"])
run(["git", "remote", "set-url", "origin", f"https://{REMOTE_REPO}"], check=False)
print("Pushed to", BRANCH)
